# Chapter 6 - Vision Language Models for Video Understanding

Companion notebook for *Video Generation with AI* (O'Reilly) by Joseph Enochs.

**Learning objective:** Understand the three-component VLM architecture (vision encoder, projector, language model) and 3D positional encoding, and build a practical video question-answering system with Qwen3-VL, including LoRA fine-tuning.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jenochs/video-ai-book-companion/blob/main/chapter-06/notebooks/chapter_6_companion_notebook.ipynb)

In [ ]:
# Core dependencies for this chapter
%pip install -q torch transformers accelerate qwen-vl-utils decord peft datasets
# Optional, speeds up attention on supported GPUs (skip on CPU-only runtimes):
# %pip install flash-attn --no-build-isolation

## Setup and run flags

The architecture and preprocessing sections below run on any Colab runtime, including CPU. Cells that download and run a full Qwen3-VL checkpoint are gated behind opt-in flags because of their size.

In [ ]:
import torch

# --- Opt-in flags for heavy model downloads ---
# RUN_MODEL_DEMOS downloads a Qwen3-VL checkpoint and runs inference.
#   Qwen3-VL-2B-Instruct: ~4 GB download, runs on a T4 (16 GB) Colab GPU.
#   Qwen3-VL-8B-Instruct: ~18 GB download, needs 18 GB+ VRAM (A100 or 4-bit quantization).
RUN_MODEL_DEMOS = False  # Set True to download and run the model

# RUN_FINETUNING runs the LoRA training loop. It needs a GPU, the model above,
# and a real video dataset in place of the toy JSON we create later.
RUN_FINETUNING = False  # Set True only with sufficient GPU memory and real data

# The book uses Qwen3-VL-8B-Instruct. We default to the smallest usable
# checkpoint so the notebook works on free Colab GPUs.
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
# MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"  # Book default, 18 GB+ VRAM

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## From Generation to Understanding

Chapters 3-5 generated video from text with diffusion models. This chapter goes the other way: a vision language model (VLM) takes pixels plus a question and produces text, turning video understanding into a language modeling problem.

| Aspect | Diffusion model (CogVideo) | VLM (Qwen3-VL) |
|---|---|---|
| Primary input | Text prompt | Video + text query |
| Output | Video frames (pixels) | Text response (tokens) |
| Core operation | Iterative denoising | Next-token prediction |
| Training signal | Noise prediction loss (MSE) | Cross-entropy loss |
| Inference steps | 20-50 denoising steps | Variable (until EOS token) |
| Output space | Continuous (latent/pixel) | Discrete (vocabulary tokens) |

## Introducing Qwen3-VL

We work with Qwen3-VL: open weights across sizes from 2B to 235B, strong video understanding, and a documented fine-tuning path. Loading the model verifies the environment; everything else in the chapter builds on this foundation.

In [ ]:
if RUN_MODEL_DEMOS:
    from transformers import AutoModelForImageTextToText, AutoProcessor

    # Load model and processor
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B parameters")
else:
    print("RUN_MODEL_DEMOS is False; skipping model download.")

## VLM Architecture: The Three-Component Model

A VLM chains three parts: a vision encoder that turns frames into feature vectors, a projector that maps those features into the language model's embedding space, and an LLM that reasons over the combined visual and text tokens.

### Vision Encoder: Seeing the Video

The Vision Transformer splits each frame into fixed-size patches (14x14 pixels in Qwen3-VL), embeds them, and runs them through transformer layers. Frames are processed independently and their patch sequences concatenated.

In [ ]:
# Conceptual token count for video processing
frame_height, frame_width = 224, 224
patch_size = 14
patches_per_frame = (frame_height // patch_size) * (frame_width // patch_size)  # 256
num_frames = 8
total_vision_tokens = patches_per_frame * num_frames  # 2,048

print(f"Patches per frame: {patches_per_frame}")
print(f"Total vision tokens for {num_frames} frames: {total_vision_tokens}")

### The Projector: Bridging Two Worlds

The vision encoder outputs one dimensionality (say 1024), the LLM expects another (say 4096). A small MLP translates between them.

In [ ]:
import torch.nn as nn

# Basic MLP projector
class Projector(nn.Module):
    def __init__(self, vision_dim, llm_dim):
        super().__init__()
        self.linear1 = nn.Linear(vision_dim, llm_dim)
        self.gelu = nn.GELU()
        self.linear2 = nn.Linear(llm_dim, llm_dim)

    def forward(self, vision_features):
        # vision_features: [batch, num_patches, vision_dim]
        x = self.linear1(vision_features)
        x = self.gelu(x)
        x = self.linear2(x)
        # output: [batch, num_patches, llm_dim]
        return x

projector = Projector(vision_dim=1024, llm_dim=4096)
dummy_features = torch.randn(1, 256, 1024)  # one frame of patch features
projected = projector(dummy_features)
print(f"Input:  {tuple(dummy_features.shape)}")
print(f"Output: {tuple(projected.shape)}")

### The Patch Merger: Cutting Token Count

Projecting all 2,048 vision tokens strains the LLM. Qwen3-VL's Patch Merger pools each 2x2 group of adjacent patches into one token before projecting, a 75% reduction.

In [ ]:
# Merger with spatial pooling
class Merger(nn.Module):
    def __init__(self, vision_dim, llm_dim, spatial_merge_size=2):
        super().__init__()
        self.merge_size = spatial_merge_size
        # Input dim multiplied by merge_size squared because we concatenate pooled patches
        merged_dim = vision_dim * (spatial_merge_size ** 2)
        self.mlp = nn.Sequential(
            nn.Linear(merged_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        )

    def forward(self, vision_features, grid_h, grid_w):
        # Reshape to spatial grid, pool adjacent patches, project
        # [batch, h*w, dim] -> [batch, h//m * w//m, dim * m^2] -> [batch, reduced, llm_dim]
        batch, _, dim = vision_features.shape
        m = self.merge_size
        x = vision_features.view(batch, grid_h, grid_w, dim)
        x = x.view(batch, grid_h // m, m, grid_w // m, m, dim)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
        x = x.view(batch, (grid_h // m) * (grid_w // m), dim * m * m)
        return self.mlp(x)

merger = Merger(vision_dim=1024, llm_dim=4096, spatial_merge_size=2)
frame_features = torch.randn(1, 16 * 16, 1024)  # 16x16 patch grid per frame
merged = merger(frame_features, grid_h=16, grid_w=16)
print(f"Tokens per frame before merging: {frame_features.shape[1]}")
print(f"Tokens per frame after merging:  {merged.shape[1]}")
print(f"8-frame video: {8 * frame_features.shape[1]} -> {8 * merged.shape[1]} tokens")

### The LLM: Reasoning Over Multimodal Tokens

The language model itself is a standard autoregressive transformer. Special tokens mark where visual content begins and ends inside the sequence:

| Token | ID | Purpose |
|---|---|---|
| `<\|vision_start\|>` | 151652 | Marks the beginning of visual content |
| `<\|vision_end\|>` | 151653 | Marks the end of visual content |
| `<\|vision_pad\|>` | 151654 | Padding within vision sequences |
| `<\|image_pad\|>` | 151655 | Placeholder for image tokens in text |
| `<\|video_pad\|>` | 151656 | Placeholder for video tokens in text |

During preprocessing, placeholder tokens in the text are replaced with projected vision tokens, and the LLM generates the answer autoregressively over the combined sequence.

## Putting It Together: A VideoQA Class

The complete pipeline, wrapped in a reusable class. `ask` builds the message, processes the vision inputs, and generates a response; `describe` and `find_event` cover common patterns.

In [ ]:
class VideoQA:
    """A video question-answering system built on Qwen3-VL."""

    def __init__(self, model_name: str = MODEL_ID):
        from transformers import AutoModelForImageTextToText, AutoProcessor
        self.model = AutoModelForImageTextToText.from_pretrained(
            model_name, torch_dtype=torch.bfloat16, device_map="auto")
        self.processor = AutoProcessor.from_pretrained(model_name)

    def ask(self, video_path: str, question: str, fps: float = 2.0,
            max_frames: int = 32, max_new_tokens: int = 512,
            system_prompt: str = None) -> str:
        """Ask a question about a video."""
        from qwen_vl_utils import process_vision_info
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": [
                {"type": "text", "text": system_prompt}]})
        messages.append({"role": "user", "content": [
            {"type": "video", "video": video_path, "fps": fps, "max_frames": max_frames},
            {"type": "text", "text": question}
        ]})
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages, image_patch_size=16)
        inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs,
                                padding=True, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        return self.processor.decode(
            outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    def describe(self, video_path: str, **kwargs) -> str:
        """Generate a description of the video."""
        return self.ask(video_path, "Describe what happens in this video.", **kwargs)

    def find_event(self, video_path: str, event: str, **kwargs) -> str:
        """Find when a specific event occurs."""
        return self.ask(video_path, f"When does: {event}? Give timestamps in MM:SS.", **kwargs)

In [ ]:
# Download a small public-domain sample clip (~2 MB) to experiment with
import urllib.request, os

SAMPLE_VIDEO = "sample_video.mp4"
if not os.path.exists(SAMPLE_VIDEO):
    url = ("https://commondatastorage.googleapis.com/gtv-videos-bucket/"
           "sample/ForBiggerEscapes.mp4")
    urllib.request.urlretrieve(url, SAMPLE_VIDEO)
print(f"{SAMPLE_VIDEO}: {os.path.getsize(SAMPLE_VIDEO) / 1e6:.1f} MB")

In [ ]:
if RUN_MODEL_DEMOS:
    qa = VideoQA()
    response = qa.ask(SAMPLE_VIDEO, "What is happening in this video?")
    print(response)

    description = qa.describe(SAMPLE_VIDEO)
    print(description)
else:
    print("RUN_MODEL_DEMOS is False; skipping VideoQA inference.")

## 3D Positional Encoding for Video

A vision token has a position in space and time, not just a spot in a sequence. Interleaved-MRoPE gives every vision token three indices: t (frame), h (row), w (column), so the model can tell the top-left of frame 1 from the bottom-right of frame 8.

In [ ]:
# Simplified, runnable version of get_rope_index_3 from Qwen3-VL's rope2d.py
def build_3d_position_ids(num_frames, grid_h, grid_w, spatial_merge_size=1):
    """
    Construct 3D position indices for vision tokens.

    Returns a tensor of shape [3, num_tokens] where:
        row 0 = temporal indices
        row 1 = height indices
        row 2 = width indices
    """
    # After spatial merging, grid dimensions are reduced
    llm_grid_t = num_frames
    llm_grid_h = grid_h // spatial_merge_size
    llm_grid_w = grid_w // spatial_merge_size

    # t_index: [0,0,0,...,0, 1,1,1,...,1, 2,2,2,...,2, ...]
    t_index = torch.arange(llm_grid_t).view(-1, 1).expand(
        -1, llm_grid_h * llm_grid_w
    ).flatten()

    # h_index: [0,0,0,1,1,1,2,2,2, 0,0,0,1,1,1,2,2,2, ...]
    h_index = torch.arange(llm_grid_h).view(1, -1, 1).expand(
        llm_grid_t, -1, llm_grid_w
    ).flatten()

    # w_index: [0,1,2,0,1,2,0,1,2, 0,1,2,0,1,2,0,1,2, ...]
    w_index = torch.arange(llm_grid_w).view(1, 1, -1).expand(
        llm_grid_t, llm_grid_h, -1
    ).flatten()

    return torch.stack([t_index, h_index, w_index])

# Concrete example: 4 frames, 3x3 spatial grid after merging (36 tokens total)
position_ids = build_3d_position_ids(num_frames=4, grid_h=3, grid_w=3)
print(f"Shape: {tuple(position_ids.shape)}  (3 axes x 36 tokens)")
print(f"t: {position_ids[0].tolist()}")
print(f"h: {position_ids[1].tolist()}")
print(f"w: {position_ids[2].tolist()}")

### Timestamps vs Frame Indices

Qwen3-VL uses timestamps rather than raw frame numbers for the temporal axis, so responses can say "at 00:02.5" instead of "in frame 5", regardless of the sampling rate.

In [ ]:
# Frame-based indexing (simpler models)
t_indices_frames = [0, 1, 2, 3, 4, 5, 6, 7]  # Just frame numbers

# Timestamp-based indexing (Qwen3-VL), video sampled at 2 FPS
t_indices_seconds = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]  # Actual seconds

for f, s in zip(t_indices_frames, t_indices_seconds):
    print(f"frame {f} -> {s:>4.1f}s")

## Extending VideoQA with Temporal Grounding

The temporal axis of the 3D encoding lets the model answer "when" questions. We parse its timestamped responses into structured segments.

In [ ]:
import re
from dataclasses import dataclass
from typing import List

@dataclass
class TemporalSegment:
    """A time segment in a video."""
    start_time: str
    end_time: str
    description: str

    def to_seconds(self, time_str: str) -> float:
        parts = time_str.split(":")
        return int(parts[0]) * 60 + float(parts[1]) if len(parts) == 2 else float(time_str)

    @property
    def duration(self) -> float:
        return self.to_seconds(self.end_time) - self.to_seconds(self.start_time)


class VideoQAWithGrounding(VideoQA):
    """Extended VideoQA with temporal grounding capabilities."""

    def ground_events(self, video_path: str, query: str, **kwargs) -> List[TemporalSegment]:
        """Find events and return structured temporal segments."""
        system_prompt = """Output events in this format:
[START_TIME-END_TIME] Description
Example: [00:05-00:12] A person enters the door
Use MM:SS format."""

        response = self.ask(video_path, query, system_prompt=system_prompt, **kwargs)

        segments = []
        pattern = r'\[(\d{2}:\d{2})-(\d{2}:\d{2})\]\s*(.+?)(?=\[|$)'
        for start, end, desc in re.findall(pattern, response, re.DOTALL):
            segments.append(TemporalSegment(start.strip(), end.strip(), desc.strip()))
        return segments

    def generate_timeline(self, video_path: str, **kwargs) -> List[TemporalSegment]:
        """Generate a complete timeline of events."""
        return self.ground_events(
            video_path, "List all significant events with timestamps.", **kwargs
        )

# The parser works standalone; try it on a sample response
sample_response = """[00:00-00:04] A woman looks out a window
[00:04-00:09] An explosion is visible outside
[00:09-00:15] She reacts and moves away from the window"""
pattern = r'\[(\d{2}:\d{2})-(\d{2}:\d{2})\]\s*(.+?)(?=\[|$)'
for start, end, desc in re.findall(pattern, sample_response, re.DOTALL):
    seg = TemporalSegment(start, end, desc.strip())
    print(f"{seg.start_time} - {seg.end_time} ({seg.duration:.0f}s): {seg.description}")

In [ ]:
if RUN_MODEL_DEMOS:
    qa = VideoQAWithGrounding()
    timeline = qa.generate_timeline(SAMPLE_VIDEO)
    for segment in timeline:
        print(f"{segment.start_time} - {segment.end_time}: {segment.description}")
else:
    print("RUN_MODEL_DEMOS is False; skipping timeline generation.")

## Video Understanding Pipeline

From file on disk to generated text: extract frames at a target FPS, smart-resize them to fit the patch grid and pixel budget, normalize, then encode, project, and generate.

### Frame Extraction

Qwen3-VL samples by FPS rather than a fixed frame count, which preserves temporal structure across videos of different lengths.

In [ ]:
import numpy as np

def fetch_video(video_path, fps=2.0, min_frames=4, max_frames=None):
    """
    Extract frames from video at specified FPS.

    Args:
        video_path: Path to video file
        fps: Target frames per second for extraction
        min_frames: Minimum frames to extract (pad if needed)
        max_frames: Maximum frames to extract (subsample if exceeded)

    Returns:
        frames: Array of shape [num_frames, height, width, channels]
    """
    from decord import VideoReader
    video = VideoReader(video_path)
    native_fps = video.get_avg_fps()
    duration = len(video) / native_fps

    # Calculate frame count based on target FPS
    num_frames = int(duration * fps)
    num_frames = max(num_frames, min_frames)
    if max_frames:
        num_frames = min(num_frames, max_frames)

    # Sample frames at regular intervals
    frame_indices = np.linspace(0, len(video) - 1, num_frames, dtype=int)
    frames = video.get_batch(frame_indices).asnumpy()
    return frames

frames = fetch_video(SAMPLE_VIDEO, fps=2.0, max_frames=32)
print(f"Extracted frames: {frames.shape}  (frames, height, width, channels)")

### Smart Resizing

Instead of forcing a fixed resolution, smart resize keeps the aspect ratio, keeps total pixels inside a budget, and rounds dimensions to multiples of the patch size.

In [ ]:
import math

def smart_resize(height, width, factor=28, min_pixels=None, max_pixels=None):
    """
    Resize dimensions to be divisible by factor while staying within pixel limits.
    Maintains aspect ratio and ensures total pixels fall within [min_pixels, max_pixels].
    """
    if min_pixels is None:
        min_pixels = 28 * 28 * 16   # ~12K pixels minimum
    if max_pixels is None:
        max_pixels = 28 * 28 * 576  # ~450K pixels maximum

    current_pixels = height * width

    # Scale down if exceeding maximum
    if current_pixels > max_pixels:
        scale = math.sqrt(max_pixels / current_pixels)
        height = int(height * scale)
        width = int(width * scale)

    # Scale up if below minimum
    if current_pixels < min_pixels:
        scale = math.sqrt(min_pixels / current_pixels)
        height = int(height * scale)
        width = int(width * scale)

    # Round to nearest multiple of factor
    height = round(height / factor) * factor
    width = round(width / factor) * factor

    # Ensure at least one patch in each dimension
    height = max(height, factor)
    width = max(width, factor)

    return height, width

# A 4K frame gets scaled down to fit the pixel budget
print(f"4K (3840x2160)  -> {smart_resize(2160, 3840)}")
print(f"720p (1280x720) -> {smart_resize(720, 1280)}")
print(f"Tiny (64x64)    -> {smart_resize(64, 64)}")

### Normalization

Pixel values are normalized with the standard ImageNet statistics used by ViT models.

In [ ]:
# Standard normalization for ViT models
mean = [0.48145466, 0.4578275, 0.40821073]
std = [0.26862954, 0.26130258, 0.27577711]

# Normalize: (pixel - mean) / std
frames_float = frames.astype(np.float32) / 255.0
normalized_frames = (frames_float - np.array(mean)) / np.array(std)
print(f"Raw pixel range:        [{frames.min()}, {frames.max()}]")
print(f"Normalized value range: [{normalized_frames.min():.2f}, {normalized_frames.max():.2f}]")

### Video Understanding Task Types

The same pipeline supports question answering, temporal grounding, and captioning. Only the text query changes.

In [ ]:
# Example: Video QA
qa_messages = [{
    "role": "user",
    "content": [
        {"type": "video", "video": SAMPLE_VIDEO},
        {"type": "text", "text": "What ingredients does the chef add to the pan?"}
    ]
}]

# Example: Temporal grounding
grounding_messages = [{
    "role": "user",
    "content": [
        {"type": "video", "video": SAMPLE_VIDEO},
        {"type": "text", "text": "When does the player score the goal?"}
    ]
}]

# Example: Video captioning
caption_messages = [{
    "role": "user",
    "content": [
        {"type": "video", "video": SAMPLE_VIDEO},
        {"type": "text", "text": "Describe what happens in this video."}
    ]
}]
print("Task templates ready: QA, temporal grounding, captioning")

### Complete Inference Example

The full pipeline in one place: load, build the message, process vision inputs, generate. This reuses the model loaded earlier when `RUN_MODEL_DEMOS` is on.

In [ ]:
if RUN_MODEL_DEMOS:
    from qwen_vl_utils import process_vision_info

    # Construct message with video
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": SAMPLE_VIDEO,
                    "fps": 2.0,           # Extract 2 frames per second
                    "max_frames": 32      # Limit total frames
                },
                {
                    "type": "text",
                    "text": "Describe what happens in this video, including timestamps."
                }
            ]
        }
    ]

    # Apply chat template and process vision inputs
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages, image_patch_size=16)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)

    # Generate and decode response
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.7)
    response = processor.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(response)
else:
    print("RUN_MODEL_DEMOS is False; skipping full inference example.")

### Handling Long Videos

Context windows are finite. Frame subsampling is the simplest fix; keyframe selection, hierarchical summarization, and sliding windows scale further.

In [ ]:
# Adaptive FPS based on video length
def adaptive_fps(duration, max_frames=128):
    """Calculate FPS to stay within max_frames."""
    if duration * 2.0 <= max_frames:
        return 2.0  # Standard rate
    return max_frames / duration  # Reduced rate

for duration in [30, 60, 300, 600]:
    print(f"{duration:>4d}s video -> sample at {adaptive_fps(duration):.2f} FPS")

# Token budget: tokens = frames x (h / patch / merge) x (w / patch / merge)
num_frames, h, w, patch, merge = 8, 448, 448, 14, 2
tokens = num_frames * (h // patch // merge) * (w // patch // merge)
print(f"\nToken budget for {num_frames} frames at {h}x{w}: {tokens} vision tokens")

## Adapting to Your Domain: Dataset Preparation

Fine-tuning data follows Qwen's conversation format: each sample pairs a video with a conversation where a `<video>` placeholder marks the insertion point. Note the format uses `from: "human"` and `from: "gpt"`, not `role: "user"` and `role: "assistant"`.

In [ ]:
import json

training_data = [
    {
        "video": "data/videos/sample_001.mp4",
        "conversations": [
            {
                "from": "human",
                "value": "<video>\nDescribe what is happening in this video."
            },
            {
                "from": "gpt",
                "value": "A person walks along a path in a park surrounded by trees."
            }
        ]
    }
]

with open("training_data.json", "w") as f:
    json.dump(training_data, f, indent=2)
print(json.dumps(training_data[0], indent=2))

### Training with LoRA

LoRA adapts the language model's attention layers, the same technique used for generation in Chapter 5, training roughly 0.5% of parameters. This cell needs a GPU, the base model, and real videos behind the paths in `training_data.json`.

In [ ]:
if RUN_FINETUNING:
    from peft import LoraConfig, get_peft_model, TaskType
    from transformers import TrainingArguments, Trainer
    from datasets import load_dataset

    # Configure LoRA: target attention layers
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=64,
        lora_alpha=128,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none"
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()  # ~0.5% trainable

    # Load dataset
    dataset = load_dataset("json", data_files="training_data.json")

    training_args = TrainingArguments(
        output_dir="./output",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        bf16=True,
        save_strategy="epoch",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"]
    )

    trainer.train()
    model.save_pretrained("./output/lora_weights")
else:
    print("RUN_FINETUNING is False; skipping LoRA training.")

### Using Fine-Tuned Models

After training, load the LoRA weights onto the base model and optionally merge them for faster inference.

In [ ]:
if RUN_FINETUNING:
    from transformers import AutoModelForImageTextToText
    from peft import PeftModel

    base_model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
    )
    model = PeftModel.from_pretrained(base_model, "./output/lora_weights")
    model = model.merge_and_unload()  # Merge for faster inference
    print("LoRA weights merged into base model.")
else:
    print("RUN_FINETUNING is False; skipping fine-tuned model loading.")

## Evaluation

A simple framework that compares model predictions against expected answers, with fuzzy matching by default.

In [ ]:
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class EvalResult:
    accuracy: float
    total: int
    correct: int
    details: List[Dict]

def evaluate_video_qa(model, test_data: List[Dict], fuzzy_match=True) -> EvalResult:
    correct = 0
    details = []
    for item in test_data:
        prediction = model.ask(item["video"], item["question"])
        expected = item["answer"]
        if fuzzy_match:
            is_correct = expected.lower() in prediction.lower()
        else:
            is_correct = prediction.strip().lower() == expected.strip().lower()
        if is_correct:
            correct += 1
        details.append({"expected": expected, "predicted": prediction, "correct": is_correct})

    return EvalResult(correct / len(test_data) if test_data else 0,
                      len(test_data), correct, details)

In [ ]:
if RUN_MODEL_DEMOS:
    test_data = [
        {"video": SAMPLE_VIDEO, "question": "Is there a person in this video?",
         "answer": "yes"},
    ]
    qa = VideoQAWithGrounding()
    results = evaluate_video_qa(qa, test_data)
    print(f"Video QA Accuracy: {results.accuracy:.1%} ({results.correct}/{results.total})")
else:
    print("RUN_MODEL_DEMOS is False; skipping evaluation run.")

## Summary

Key ideas from this chapter:

- VLMs turn video understanding into language modeling: predict text tokens conditioned on visual input rather than predicting pixels.
- The three-component architecture (vision encoder, projector, LLM) enables modular training and transfer learning from pretrained parts.
- 3D positional encoding with separate temporal, height, and width indices is what makes spatial-temporal reasoning possible.
- The LoRA techniques from Chapter 5 carry over directly for domain adaptation.
- Production quality depends on preprocessing: frame sampling rate, smart resizing, and token budget management.

Chapter 7 adds the missing modality, audio, and covers audio-visual synchronization with cross-modal fusion.